In [2]:
#Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
openai = OpenAI()

In [5]:
# Some lists!

todos = []
completed = []

In [6]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [7]:
get_todo_report()

''

In [8]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [9]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [10]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [11]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [12]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [13]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [14]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [18]:
system_message = """
You are a travel planning agent.

You must:
1. Create a structured todo plan using your tools.
2. Plan a visit to Disneyland for New Year.
3. Assume the user lives within an hour away.
4. Estimate realistic event timings.
5. Plan food options.
6. Include evening activities.
7. Mark each step complete after executing it.
8. Provide a final polished itinerary in Rich console markup.

Do not ask the user any questions.
If information is missing, make reasonable assumptions.
Respond only with the final itinerary after using your tools.
"""

In [20]:
user_message = """
I am visiting DisneyLand this December for New Year.
Plan my visit including:
- Travel time 
- Best time to see parade 
- Food options
- Evening activities
Assume I live within 2 miles.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

In [21]:
todos, completed = [], []
loop(messages)

Todo #1: Draft structured to-do plan for Disneyland New Year visit (assume 2 miles away).
Todo #2: Set realistic travel/arrival timing and entry strategy for Dec 31.
Todo #3: Schedule key park activities incl. best parade viewing time/location and rides with lower waits.
Todo #4: Plan food options (breakfast/lunch/dinner/snacks) with timing and backups.
Todo #5: Plan evening activities incl. fireworks/countdown, entertainment, and exit/return home timing.
Todo #6: Produce final polished Rich console markup itinerary; mark all steps complete.

Created planning framework: set assumptions (Dec 31, very high crowds, user is ~2 miles away ~10–20 min 
drive/ride-share), built day structure (rope drop → midday breaks → parade → dinner → fireworks/countdown → exit).

Todo #1: Draft structured to-do plan for Disneyland New Year visit (assume 2 miles away).
Todo #2: Set realistic travel/arrival timing and entry strategy for Dec 31.
Todo #3: Schedule key park activities incl. best parade viewing time/location and rides with lower waits.
Todo #4: Plan food options (breakfast/lunch/dinner/snacks) with timing and backups.
Todo #5: Plan evening activities incl. fireworks/countdown, entertainment, and exit/return home timing.
Todo #6: Produce final polished Rich console markup itinerary; mark all steps complete.

Travel/arrival plan set: target early arrival to beat security/turnstiles; included buffer for NYE crowd control, 
parking vs rideshare, and mobile-order windows.

Todo #1: Draft structured to-do plan for Disneyland New Year visit (assume 2 miles away).
Todo #2: Set realistic travel/arrival timing and entry strategy for Dec 31.
Todo #3: Schedule key park activities incl. best parade viewing time/location and rides with lower waits.
Todo #4: Plan food options (breakfast/lunch/dinner/snacks) with timing and backups.
Todo #5: Plan evening activities incl. fireworks/countdown, entertainment, and exit/return home timing.
Todo #6: Produce final polished Rich console markup itinerary; mark all steps complete.

...

Evening plan built: fireworks viewing strategy + post-fireworks movement, countdown options, late-night rides, and 
realistic exit timing with crowd egress buffer.

Todo #1: Draft structured to-do plan for Disneyland New Year visit (assume 2 miles away).
Todo #2: Set realistic travel/arrival timing and entry strategy for Dec 31.
Todo #3: Schedule key park activities incl. best parade viewing time/location and rides with lower waits.
Todo #4: Plan food options (breakfast/lunch/dinner/snacks) with timing and backups.
Todo #5: Plan evening activities incl. fireworks/countdown, entertainment, and exit/return home timing.
Todo #6: Produce final polished Rich console markup itinerary; mark all steps complete.

Final itinerary assembled in Rich console markup with timeline, locations, and contingencies; all steps completed.

Todo #1: Draft structured to-do plan for Disneyland New Year visit (assume 2 miles away).
Todo #2: Set realistic travel/arrival timing and entry strategy for Dec 31.
Todo #3: Schedule key park activities incl. best parade viewing time/location and rides with lower waits.
Todo #4: Plan food options (breakfast/lunch/dinner/snacks) with timing and backups.
Todo #5: Plan evening activities incl. fireworks/countdown, entertainment, and exit/return home timing.
Todo #6: Produce final polished Rich console markup itinerary; mark all steps complete.

```rich
Disneyland (Anaheim) — New Year’s Eve Visit Plan (December 31)
Assumptions: You live ~2 miles away (10–20 min door-to-gate depending on traffic/security). NYE is peak-crowd; plan
extra buffers and use Mobile Order.

Quick strategy highlights
• Travel time: 10–20 min travel + 20–45 min security/turnstiles (longer later in day).
• Best parade viewing: Stake out 45–60 min early. Best “balanced” spot = Town Square / Main Street hub (great 
atmosphere + easy post-parade move).
• Food: Use Mobile Order for quick-service; schedule one “real meal” before 6:30–7:00 PM.
• Evening activities: Fireworks + countdown vibe + late-night rides; expect slow exit after midnight.

──────────────────────────────────────────────────────────────────────────────
Timeline Itinerary (Realistic NYE Timing)

7:15 AM — Leave home
• If driving: head to Mickey & Friends / Pixar Pals early to reduce entry waits.
• If rideshare: plan drop at Harbor Blvd area; walking/security can still be the bottleneck.

7:35–8:15 AM — Security + entry + “rope drop” positioning
• Have tickets/park reservation ready (if required), bags simplified, pockets empty for security speed.
• Tip: open the Disneyland app immediately for wait times + Mobile Order windows.

8:15–10:30 AM — Hit headliners while waits are lowest
Pick 3–4 depending on preference:
• Fantasyland classics (shortest early): Peter Pan, Alice in Wonderland, Matterhorn
• Tomorrowland: Space Mountain early if it’s a must-do
• Frontierland/New Orleans: Big Thunder, Pirates (often manageable early)

Food option (Breakfast, ~9:30–10:15 AM)
• Quick + iconic: Jolly Holiday Bakery Cafe (pastries/sandwiches)
• Hearty: Red Rose Taverne
• Grab-and-go backup: fruit + breakfast items from any quick-service, eat while walking to your next land.

10:30 AM–12:30 PM — Attractions + “flex time”
• Focus on dark rides/indoor queues while crowds build.
• If you want photos: do them now (castle/Main Street) before it’s shoulder-to-shoulder.

Lunch (12:00–1:30 PM)
Primary (Mobile Order recommended):
• Plaza Inn (fried chicken; filling “big meal”)
• Bengal Barbecue (skewers; quick + high-protein)
Backups:
• Docking Bay 7 (Galaxy’s Edge; good variety)
• Rancho del Zocalo (easy seating when you find it)

1:30–3:30 PM — Midday plan: lower-stress block
• Do lower-wait/people-eater attractions: Haunted Mansion, it’s a small world, Winnie the Pooh
• Optional: take a 30–45 min “reset” break (coffee, sit-down, hydrate). NYE stamina matters.

Parade Block (Best time + how to do it)
3:30–4:15 PM — Start migrating to your parade-view area
4:15–5:15 PM — Stake out your spot (45–60 min is realistic on NYE)

Best parade viewing recommendation
• Town Square / Main Street hub
  - Pros: biggest “Disney magic” feel, great sightlines, easiest to flow into shops/food after.
  - Cons: busiest; arrive early.
• Alternate “less packed” option: near it’s a small world parade viewing area
  - Pros: more breathing room, easier to sit.
  - Cons: you’ll travel farther afterward for evening plans.

~5:30 PM — Parade (typical afternoon/early evening window)
Exact time can vary by day/season; this plan builds in a realistic stakeout buffer.

Dinner (6:00–7:15 PM)
Goal: eat before the night crowds fully lock in.

Quick-service (best for speed):
• Tiana’s Palace (New Orleans Square; great flavors)
• Galactic Grill (Tomorrowland; convenient, fast)
• Hungry Bear Barbecue Jamboree (Critter Country area; solid “comfort” option)

If you want sit-down (booked/limited on NYE, but plan as an option):
• Café Orleans or Blue Bayou (if you have a reservation)
Backups if lines are wild:
• Corn dogs (Little Red Wagon on Main Street)
• Soup in a bread bowl (when available nearby)
• Stock up: bottle of water + a portable snack for the late night

7:15–8:45 PM — Evening attractions + positioning
• Do attractions with strong nighttime vibe: Pirates, Big Thunder, Haunted Mansion
• Start thinking about fireworks/crowd-control: moving gets slower after ~8:30 PM.

Fireworks Plan (Ev

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>